# 03 · Phân tích truy xuất — Recall@k và ngưỡng

**Máy:** CPU · **Thời gian:** ~2–4 phút · **Chi phí API:** 0 đồng

Notebook này **không nhúng lại gì cả** — nó nạp `emb.npy` từ notebook 02. Nhờ vậy
chỉnh biểu đồ bao nhiêu lần cũng không tốn hạn mức GPU.

## Chuẩn bị

Add Input → (1) dataset `longmemeval`, (2) output notebook 01, (3) output notebook 02.

## Notebook này trả lời ba câu quyết định thiết kế

| Câu hỏi | Trả lời bằng |
|---|---|
| Đặt ngưỡng từ chối lên cosine được không? | Hai phân bố cosine chồng nhau bao nhiêu |
| BM25 và RRF có đáng thêm vào không? | Đường Recall@k, ba phương pháp |
| Nên đầu tư vào loại câu hỏi nào? | Recall@10 tách theo loại |

> Toàn bộ phân tích chạy trên **`dev`**. `test` vẫn khóa.

In [ ]:
!pip install -q rank_bm25
print("xong")

In [ ]:
# ============ Cấu hình ============
from pathlib import Path
import json, re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from rank_bm25 import BM25Okapi

DATA_DIR = Path("/kaggle/input/longmemeval")
NB01_DIR = Path("/kaggle/input/longmemeval-01-explore")   # sửa cho khớp
NB02_DIR = Path("/kaggle/input/longmemeval-02-embed")     # sửa cho khớp
OUT_DIR  = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True)

SPLIT   = "dev"                       # 'dev' | 'test' | 'all'  — để nguyên 'dev'
K_LIST  = [1, 2, 3, 5, 10, 20, 30, 50]
RRF_K   = 60
POOL    = 100                         # mỗi phương pháp đưa 100 ứng viên vào RRF

In [ ]:
# ============ Bảng màu đã kiểm định (giống notebook 01) ============
P1, P2, P3 = "#5B57C9", "#C77F06", "#00908C"
INK, INK2, MUTED, GRID = "#1A1A1A", "#4A4A4A", "#767676", "#E4E4E4"

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 180, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.labelcolor": INK2, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlecolor": INK, "axes.titlelocation": "left", "axes.titlepad": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "grid.color": GRID, "grid.linewidth": 0.7, "legend.frameon": False,
    "font.size": 10, "text.color": INK2,
})

def finish(ax, title, sub=None, axis="y"):
    ax.set_title(title)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9,
                color=MUTED, va="bottom")
    ax.set_axisbelow(True); ax.grid(axis=axis)
    return ax

def save(fig, name):
    fig.tight_layout(); fig.savefig(OUT_DIR / f"{name}.png", bbox_inches="tight")
    print("đã lưu:", name + ".png")

In [ ]:
# ============ Nạp ============
emb      = np.load(NB02_DIR / "emb.npy")
texts    = pd.read_parquet(NB02_DIR / "texts.parquet")
manifest = json.load(open(NB02_DIR / "manifest.json"))
split    = json.load(open(NB01_DIR / "split.json"))

print(json.dumps(manifest, indent=2))
print(f"\nemb {emb.shape} · texts {len(texts):,} dòng")

if SPLIT == "all":
    qids = sorted(texts.loc[texts.kind == "question", "question_id"].unique())
else:
    want = set(split[SPLIT])
    have = set(texts.question_id.unique())
    qids = sorted(want & have)

print(f"\nPhân tích trên '{SPLIT}': {len(qids)} câu hỏi")
if manifest.get("smoke"):
    print("CẢNH BÁO: embedding đang ở chế độ thử (SMOKE=True) — số liệu chưa đầy đủ.")

In [ ]:
# ============ Gom dữ liệu theo từng câu hỏi ============
by_q = {}
for qid, g in texts[texts.question_id.isin(qids)].groupby("question_id", sort=False):
    q_row  = g[g.kind == "question"]
    turns  = g[g.kind == "turn"]
    if not len(q_row) or not len(turns):
        continue
    pos = turns.has_answer.values
    if pos.sum() == 0:            # không có bằng chứng được gắn cờ → bỏ
        continue
    by_q[qid] = {
        "q_row":  int(q_row.row_id.iloc[0]),
        "q_text": q_row.text.iloc[0],
        "rows":   turns.row_id.values,
        "texts":  turns.text.tolist(),
        "pos":    pos,
    }

print(f"{len(by_q)} câu hỏi có đủ bằng chứng để phân tích")
qtype = (pd.read_parquet(NB01_DIR / "questions.parquet")
           .set_index("question_id")["question_type"].to_dict())

## C1 · Có đặt ngưỡng lên cosine được không?

Vẽ **hai phân bố chồng lên nhau**: cosine giữa câu hỏi và câu *chứa đáp án*, so với
cosine giữa câu hỏi và một lượt *ngẫu nhiên* trong cùng haystack.

Hai phân bố chồng nhau càng nhiều thì càng **không** có ngưỡng nào tách được chúng —
đó là lập luận bằng số cho việc phải dùng điểm của reranker thay vì cosine.

In [ ]:
rng = np.random.default_rng(0)
s_pos, s_neg = [], []

for qid, d in by_q.items():
    qv = emb[d["q_row"]].astype(np.float32)
    sims = emb[d["rows"]].astype(np.float32) @ qv        # đã chuẩn hóa L2 → = cosine
    s_pos.extend(sims[d["pos"]].tolist())
    neg = sims[~d["pos"]]
    if len(neg):
        s_neg.extend(rng.choice(neg, min(len(neg), 30), replace=False).tolist())

s_pos, s_neg = np.array(s_pos), np.array(s_neg)

# hệ số chồng lấn: tổng phần chung của hai histogram đã chuẩn hóa
lo, hi = min(s_pos.min(), s_neg.min()), max(s_pos.max(), s_neg.max())
bins = np.linspace(lo, hi, 60)
h_pos, _ = np.histogram(s_pos, bins=bins, density=True)
h_neg, _ = np.histogram(s_neg, bins=bins, density=True)
overlap = float(np.minimum(h_pos, h_neg).sum() * (bins[1] - bins[0]))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(s_neg, bins=bins, density=True, color=MUTED, alpha=0.45,
        label="lượt ngẫu nhiên", edgecolor="white", linewidth=0.4)
ax.hist(s_pos, bins=bins, density=True, color=P1, alpha=0.75,
        label="câu chứa đáp án", edgecolor="white", linewidth=0.4)
ax.axvline(s_neg.mean(), color=MUTED, linewidth=1.6, linestyle="--")
ax.axvline(s_pos.mean(), color=P1, linewidth=2)
ax.text(s_pos.mean(), ax.get_ylim()[1] * 0.95, f"  TB {s_pos.mean():.3f}",
        color=P1, fontsize=9, fontweight="bold")
ax.text(s_neg.mean(), ax.get_ylim()[1] * 0.82, f"  TB {s_neg.mean():.3f}",
        color=MUTED, fontsize=9)
ax.set_xlabel("cosine giữa câu hỏi và đoạn văn bản"); ax.set_ylabel("mật độ")
ax.legend(loc="upper left")
finish(ax, "Cosine không tách được liên quan khỏi không liên quan",
       f"phần chồng lấn {overlap:.1%} — ngưỡng cắt ở đâu cũng sai một lượng lớn")
save(fig, "c1_phan_bo_cosine"); plt.show()

print(f"Chồng lấn {overlap:.1%}.")
print(f"Cosine dương tính: TB {s_pos.mean():.3f}, p10 {np.percentile(s_pos,10):.3f}")
print(f"Cosine âm tính   : TB {s_neg.mean():.3f}, p90 {np.percentile(s_neg,90):.3f}")
print("\n→ Đây là số liệu để nói với mentor vì sao ngưỡng từ chối phải đặt lên "
      "điểm cross-encoder chứ không phải cosine.")

## C2 · Recall@k — BM25 và RRF có đáng không?

**Đây là biểu đồ đáng giá nhất của cả notebook.** Nó đo *thuần truy xuất*, tách khỏi
phần sinh, và **không tốn một lời gọi LLM nào**.

- `recall@k` — tỉ lệ câu chứa đáp án lọt vào top-k
- Ba đường: chỉ vector · chỉ BM25 · hợp nhất RRF

Nếu đường RRF không nằm trên hai đường kia thì truy xuất lai **không đáng làm**, và
nhóm sẽ bỏ nó — báo cáo đúng như vậy.

In [ ]:
STOP = set("""a an the is are was were be been being of in on at to for with and or
but if then than that this these those i you he she it we they my your his her its
our their me him them do does did have has had can could will would shall should
may might must not no nor so such as by from about into over after before""".split())

def toks(s):
    return [w for w in re.findall(r"[a-z0-9]+", s.lower())
            if len(w) > 1 and w not in STOP]

def rrf_fuse(rank_lists, k=RRF_K):
    """rank_lists: list các mảng chỉ số, đã sắp theo thứ hạng tốt→xấu."""
    score = defaultdict(float)
    for lst in rank_lists:
        for rank, idx in enumerate(lst, start=1):
            score[idx] += 1.0 / (k + rank)
    return np.array(sorted(score, key=score.get, reverse=True))

recs = []
for n, (qid, d) in enumerate(by_q.items(), 1):
    pos_idx = set(np.where(d["pos"])[0].tolist())
    n_pos   = len(pos_idx)

    qv   = emb[d["q_row"]].astype(np.float32)
    sims = emb[d["rows"]].astype(np.float32) @ qv
    r_dense = np.argsort(-sims)

    corpus  = [toks(t) for t in d["texts"]]
    bm      = BM25Okapi(corpus)
    r_bm25  = np.argsort(-bm.get_scores(toks(d["q_text"])))

    r_rrf   = rrf_fuse([r_dense[:POOL], r_bm25[:POOL]])

    row = {"question_id": qid, "question_type": qtype.get(qid, "?"),
           "n_cand": len(d["texts"]), "n_pos": n_pos}
    for name, order in [("dense", r_dense), ("bm25", r_bm25), ("rrf", r_rrf)]:
        for k in K_LIST:
            top = set(order[:k].tolist())
            row[f"recall@{k}_{name}"] = len(pos_idx & top) / n_pos
            row[f"hit@{k}_{name}"]    = float(len(pos_idx & top) > 0)
    recs.append(row)

    if n % 50 == 0:
        print(f"  {n}/{len(by_q)}", flush=True)

R = pd.DataFrame(recs)
print(f"\nXong {len(R)} câu hỏi · trung vị {R.n_cand.median():.0f} ứng viên/câu")
R.to_parquet(OUT_DIR / "recall.parquet", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))

series = [("dense", "Vector (BGE-M3)", P1),
          ("bm25",  "BM25 từ khóa",    P2),
          ("rrf",   "Hợp nhất RRF",    P3)]

for key, label, color in series:
    ys = [R[f"recall@{k}_{key}"].mean() for k in K_LIST]
    ax.plot(K_LIST, ys, color=color, linewidth=2, marker="o",
            markersize=5, markeredgecolor="white", markeredgewidth=1.2,
            label=label, zorder=3)
    ax.annotate(f" {label} · {ys[-1]:.0%}", (K_LIST[-1], ys[-1]),
                color=color, fontsize=9, fontweight="bold",
                va="center", xytext=(6, 0), textcoords="offset points")

ax.set_xscale("log"); ax.set_xticks(K_LIST)
ax.set_xticklabels([str(k) for k in K_LIST])
ax.set_xlim(K_LIST[0] * 0.85, K_LIST[-1] * 2.6)
ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("k — số ứng viên lấy ra"); ax.set_ylabel("recall trung bình")
ax.legend(loc="lower right")
finish(ax, "Recall@k — đo thuần truy xuất, không tốn lời gọi LLM",
       f"{len(R)} câu hỏi trên tập {SPLIT} · trung vị {R.n_cand.median():.0f} ứng viên mỗi câu")
save(fig, "c2_recall_at_k"); plt.show()

tab = pd.DataFrame({lbl: [R[f"recall@{k}_{key}"].mean() for k in K_LIST]
                    for key, lbl, _ in series}, index=[f"k={k}" for k in K_LIST])
display((tab * 100).round(1))

g30 = tab.loc["k=30"]
print(f"\nTại k=30: RRF hơn chỉ-vector {g30['Hợp nhất RRF'] - g30['Vector (BGE-M3)']:+.1%}, "
      f"hơn chỉ-BM25 {g30['Hợp nhất RRF'] - g30['BM25 từ khóa']:+.1%}")
print("→ Nếu mức hơn này nhỏ hơn 2–3 điểm thì truy xuất lai không đáng "
      "thêm độ phức tạp; báo cáo trung thực và bỏ nó.")

## C3 · Loại câu hỏi nào truy xuất trượt nhiều nhất

Biết chỗ yếu để đầu tư đúng chỗ, thay vì tối ưu đều tay.

In [ ]:
K_SHOW = 10
types = (R.groupby("question_type")[f"recall@{K_SHOW}_rrf"]
           .mean().sort_values().index.tolist())
y = np.arange(len(types)); h = 0.26

fig, ax = plt.subplots(figsize=(10, max(3.4, 0.62 * len(types) + 1.4)))
for i, (key, label, color) in enumerate(series):
    vals = [R.loc[R.question_type == t, f"recall@{K_SHOW}_{key}"].mean() for t in types]
    ax.barh(y + (1 - i) * h, vals, height=h * 0.92, color=color, label=label)
    if key == "rrf":                                   # nhãn trực tiếp cho đường chính
        for yy, v in zip(y + (1 - i) * h, vals):
            ax.text(v + 0.012, yy, f"{v:.0%}", va="center",
                    fontsize=8.5, color=color, fontweight="bold")

ax.set_yticks(y); ax.set_yticklabels(types, fontsize=9)
ax.set_xlim(0, 1.08); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel(f"recall@{K_SHOW}")
ax.legend(loc="lower right")
finish(ax, f"Recall@{K_SHOW} theo loại câu hỏi",
       "loại nằm trên cùng là chỗ truy xuất yếu nhất", axis="x")
save(fig, "c3_recall_theo_loai"); plt.show()

display((R.groupby("question_type")[[f"recall@{K_SHOW}_{k}" for k, _, _ in series]]
           .mean() * 100).round(1)
          .rename(columns=lambda c: c.replace(f"recall@{K_SHOW}_", "")))

## Kết luận rút ra cho thiết kế

Điền vào sau khi chạy — đây là những câu sẽ nói với mentor:

| Câu hỏi thiết kế | Số liệu trả lời |
|---|---|
| Ngưỡng từ chối đặt lên cosine được không? | Chồng lấn **___%** → không; phải dùng điểm reranker |
| BM25 có đáng thêm? | RRF hơn chỉ-vector **___** điểm tại k=30 |
| `QueryPlanner` có đáng làm? | **___%** số câu cần >1 phiên bằng chứng *(notebook 01)* |
| k nên đặt bao nhiêu? | Recall bão hòa quanh k = **___** |
| Đầu tư vào đâu? | Loại **___** có recall thấp nhất |

**Lưu ý khi báo cáo:** toàn bộ số ở đây là **recall của tầng truy xuất**, chưa phải
accuracy cuối. Một câu có bằng chứng trong top-k vẫn có thể bị trả lời sai ở pha sinh.
Chênh lệch giữa hai con số đó chính là thứ baseline `oracle` sẽ đo ở tuần 7.

In [ ]:
# ============ Xuất số liệu cho báo cáo ============
out = {
    "split": SPLIT,
    "n_questions": int(len(R)),
    "embedding_model": manifest["model"],
    "cosine_overlap": round(overlap, 4),
    "cosine_pos_mean": round(float(s_pos.mean()), 4),
    "cosine_neg_mean": round(float(s_neg.mean()), 4),
    "recall": {lbl: {f"k={k}": round(float(R[f"recall@{k}_{key}"].mean()), 4)
                     for k in K_LIST}
               for key, lbl, _ in series},
}
with open(OUT_DIR / "retrieval_findings.json", "w") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

print(json.dumps(out, indent=2, ensure_ascii=False))
print("\nCác file đã lưu:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:28s} {p.stat().st_size/1e3:8.1f} KB")